# MCDI500 — Programación para la Ciencia de Datos
## Fase 1: Definición del problema y configuración del entorno

**Proyecto:** PROYECTO-GRUPO4-MCDI500  
**Integrantes:**
- Carolina Cortés Donoso
- Pedro Espinoza Vicentela
- Marcelo Corro Troncoso
- Juan Pablo Valdebenito Loyola

**Docente:** Omar Iván Salinas Silva  
**Fecha:** Junio 2026

---
## 1. Contextualización del problema

El cáncer de mama es una de las principales causas de muerte oncológica en mujeres a nivel mundial. Según la Organización Mundial de la Salud (OMS), en 2022 se diagnosticaron aproximadamente 2,3 millones de nuevos casos y cerca de 670.000 mujeres fallecieron por esta enfermedad.

Dado que los principales factores de riesgo (género femenino, edad mayor a 40 años) no pueden ser controlados, la detección temprana se convierte en la estrategia más efectiva para reducir la mortalidad. En este contexto, el análisis de datos y las técnicas de aprendizaje automático ofrecen una oportunidad para apoyar el diagnóstico médico identificando patrones morfológicos asociados a tumores malignos.

### Pregunta central de investigación

> **¿Es posible predecir de manera precisa si un tumor mamario es benigno o maligno utilizando únicamente características morfológicas celulares?**

---
## 2. Objetivos del proyecto

### Objetivo general
Desarrollar y evaluar modelos de aprendizaje automático que permitan clasificar tumores mamarios como benignos o malignos a partir de características morfológicas celulares, identificando los factores de mayor influencia en la clasificación.

### Objetivos específicos
1. Comprender la estructura y características del dataset Breast Cancer Wisconsin (Diagnostic).
2. Configurar un entorno de trabajo reproducible con las herramientas del ecosistema científico de Python.
3. Realizar la limpieza, transformación y análisis exploratorio de los datos (F2).
4. Entrenar y comparar modelos de clasificación supervisada: Regresión Logística y Random Forest (F3).
5. Evaluar el desempeño de los modelos mediante métricas apropiadas e interpretar los resultados (F3).

### Alcance
**Incluye:** obtención y comprensión del dataset, limpieza y preparación de datos, análisis exploratorio, entrenamiento y evaluación de modelos, comparación de algoritmos, visualización de métricas.  
**No incluye:** uso clínico real del modelo, diagnóstico médico de pacientes, integración a sistemas de salud.

---
## 3. Dataset: Breast Cancer Wisconsin (Diagnostic)

El dataset fue creado por el Dr. William H. Wolberg (Universidad de Wisconsin) y contiene registros de biopsias realizadas mediante aspiración con aguja fina (FNA) sobre masas mamarias.

| Característica | Detalle |
|---|---|
| Instancias | 569 |
| Variables | 32 (ID, diagnóstico, 30 características morfológicas) |
| Variable objetivo | Diagnosis: B = benigno, M = maligno |
| Distribución | 357 benignos, 212 malignos |
| Valores faltantes | Ninguno |

Las 30 variables numéricas describen características de los núcleos celulares: radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal. Cada una se mide como media, error estándar y valor máximo ("worst").

---
## 4. Configuración del entorno y verificación de dependencias

In [ ]:
# Verificación de versiones del entorno
import sys
import numpy as np
import pandas as pd
import matplotlib
import seaborn as sns
import sklearn

print(f"Python      : {sys.version}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print(f"Seaborn     : {sns.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")

---
## 5. Carga del dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Nombres de columnas según wdbc.names
COLUMNAS = [
    'id', 'diagnostico',
    'radio_mean', 'textura_mean', 'perimetro_mean', 'area_mean',
    'suavidad_mean', 'compacidad_mean', 'concavidad_mean', 'puntos_concavos_mean',
    'simetria_mean', 'dimension_fractal_mean',
    'radio_se', 'textura_se', 'perimetro_se', 'area_se',
    'suavidad_se', 'compacidad_se', 'concavidad_se', 'puntos_concavos_se',
    'simetria_se', 'dimension_fractal_se',
    'radio_worst', 'textura_worst', 'perimetro_worst', 'area_worst',
    'suavidad_worst', 'compacidad_worst', 'concavidad_worst', 'puntos_concavos_worst',
    'simetria_worst', 'dimension_fractal_worst'
]

# Ruta relativa al dataset
RUTA_DATASET = Path('../../data/raw/wdbc.data')

df = pd.read_csv(RUTA_DATASET, header=None, names=COLUMNAS)
print(f"Dataset cargado correctamente: {df.shape[0]} filas, {df.shape[1]} columnas")

---
## 6. Exploración inicial del dataset

In [ ]:
def explorar_dataset(df: pd.DataFrame) -> None:
    """Imprime un resumen estructural del dataset."""
    print("=== Primeras filas ===")
    display(df.head())

    print("\n=== Dimensiones ===")
    print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

    print("\n=== Tipos de datos ===")
    print(df.dtypes)

    print("\n=== Valores faltantes ===")
    faltantes = df.isnull().sum()
    print(faltantes[faltantes > 0] if faltantes.any() else "No hay valores faltantes.")

    print("\n=== Distribución de la variable objetivo ===")
    distribucion = df['diagnostico'].value_counts()
    print(distribucion)
    print(f"\nBenignos (B): {distribucion.get('B', 0)} | Malignos (M): {distribucion.get('M', 0)}")

explorar_dataset(df)

In [ ]:
# Estadísticas descriptivas de las variables numéricas
print("=== Estadísticas descriptivas ===")
display(df.describe().T)

---
## 7. Visualización inicial de la distribución del diagnóstico

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def graficar_distribucion_diagnostico(df: pd.DataFrame) -> None:
    """Genera un gráfico de barras con la distribución de la variable objetivo."""
    conteo = df['diagnostico'].value_counts()
    etiquetas = ['Benigno (B)', 'Maligno (M)']

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(x=etiquetas, y=conteo.values, palette=['steelblue', 'tomato'], ax=ax)

    for i, v in enumerate(conteo.values):
        ax.text(i, v + 5, str(v), ha='center', fontweight='bold')

    ax.set_title('Distribución de diagnósticos en el dataset', fontsize=13)
    ax.set_xlabel('Diagnóstico')
    ax.set_ylabel('Cantidad de casos')
    plt.tight_layout()
    plt.show()

graficar_distribucion_diagnostico(df)

---
## 8. Conclusiones de la Fase 1

En esta primera fase se logró:

- **Definir la problemática** y la pregunta de investigación que guía el proyecto.
- **Configurar el entorno reproducible** con las librerías necesarias para las fases siguientes.
- **Cargar y explorar el dataset** Breast Cancer Wisconsin (Diagnostic), confirmando su estructura: 569 instancias, 30 variables morfológicas y una variable objetivo binaria sin valores faltantes.
- **Verificar la distribución de clases**: 357 casos benignos (62.7%) y 212 malignos (37.3%), lo que indica un dataset moderadamente balanceado.

### Proyección hacia fases siguientes
- **F2:** Preprocesamiento, limpieza, normalización y análisis exploratorio profundo.
- **F3:** Entrenamiento, evaluación e interpretación de modelos (Regresión Logística y Random Forest).

---
## Referencias

- Wolberg, W. H., Street, W. N., & Mangasarian, O. L. (1994). *Machine learning techniques to diagnose breast cancer from fine-needle aspirates*. Cancer Letters, 77, 163–171.
- UCI Machine Learning Repository. (1995). *Breast Cancer Wisconsin (Diagnostic)*. https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic
- Organización Mundial de la Salud. (2026). *Cáncer de mama*. https://www.who.int/es/news-room/fact-sheets/detail/breast-cancer